# Week 3 - Classification Lesson

In [1]:
## 3.1. PROJECT OVERVIEW - PREDICT CUSTOMER CHURN AT A TELCOM COMPANY

## 3.2. DATA PREPARATION

In [2]:
# 3.1. IMPORT PACKAGES
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

In [3]:
# 3.2 DOWNLOAD AND PREVIEW DATA
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data)
df.head()

# !wget $data -O data-week-3.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# 3.3. STANDARDIZE COLUMN NAMES
# Preview column names
print(f"Column Names Before:\r\n {df.columns}")

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# Preview column names
print()
print(f"Column Names After:\r\n {df.columns}")

Column Names Before:
 Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Column Names After:
 Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')


In [5]:
# 3.4 REVIEW pandas ASSIGNED COLUMN DATA TYPES
df.dtypes


customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

In [6]:
# 'seniorcitizen' COLUMN HOLDS ONLY 1 AND 0 VALUES AS IF THIS IS A TRUE/FALSE VALUE
# WE CAN TREAT THIS AS A CATEGORICAL VALUE LATER

# 'totalcharges' WAS MISTYPED AS A STRING WHEN IT CLEARLY CONTAINS NUMERIC VALUES
# WE SHOULD CONVERT THIS TO A NUMERIC FIELD

# USE THE to_numeric() METHOD OF THE pandas PACKAGE TO CONVERT 'totalcharges' COLUMN 
# FROM STRING OBJECT TO NUMERIC
pd.to_numeric(df.totalcharges)

#   NOTE: THIS SHOULD THROW AN ERROR

ValueError: Unable to parse string "_" at position 488

In [ ]:
# THIS FAILED BECAUSE THERE WERE EMPTY SPACES IN THE RAW DATA IN THIS COLUMN
#  RECALL THAT WE REPLACED ALL SPACE CHARACTERS WITH UNDERSCORES, 
#  ERROR OCCURS BECAUSE 'pandas' IS UNABLE TO CONVERT AN UNDERSCORE TO A NUMBER

In [ ]:
# CONVERT NUMERIC STRINGS TO NUMBERS - PART 1
#   STEP 1: USE THE 'errors='coerce' PARAMETER TO REPLACE CONVERSION ERRORS WITH 'NaN'
tc = pd.to_numeric(df.totalcharges, errors='coerce')

#   STEP 2: COUNT 'NaN' ERRORS
print(f"tc.isnull().sum()\t{tc.isnull().sum()}")

#   STEP 3: INSPECT 'NaN' records
#    df                              # Full dataframe
#    tc                              # dataframe of indices where the 'tc.isnull()' 
#                                    #  test returns a 'true'
#    ['customerid', 'totalcharges']  # List of columns to return from dataframe 'df'
df[tc.isnull()][['customerid', 'totalcharges']]

In [ ]:
# CONVERT NUMERIC STRINGS TO NUMBERS - PART 2
#  SINCE 'totalcharges' REPRESENTS CUSTOMERS WHO JUST JOINED, 
#  THE INITIAL BALANCE SHOULD BE ZERO
#    WE SHOULD REPLACE 'NaN' VALUES WITH ZEROS
#    THIS MAKES SENSE BECAUSE NEW CUSTOMERS HAVE NOT PAID ANYTHING YET
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

In [ ]:
# PREVIEW 'churn' CATEGORICAL COLUMN CONTENTS
df.churn.head()

In [ ]:
# CONVERT 'churn' CATEGORICAL VALUE
#  CONVERT 'yes' AND 'no' VALUES TO INTEGERS
#  df.churn column will be updated
#  where df.churn == 'yes', update to '1', otherwise update to '0'
df.churn = (df.churn == 'yes').astype(int)

## 3.3. VALIDATION

In [ ]:
# IMPORT train-test-split FROM PACKAGE sklearn
from sklearn.model_selection import train_test_split

In [ ]:
# SET ASIDE 20% FOR THE 'df_test' TEST DATASET
#   df_full_train    - name of dataset receive the bulk of the records taken from the full dataset
#   df_test          - name of remainder dataset to receive the remainder of the records from the full dataset
#   train_test_split - method within the 'model_selection' namespace of the 'sklearn' package
#   df               - dataframe containing all records
#   test_size        - percentage of full dataset to place into the remainder dataset
#   random_state     - seed for the random number generator
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)



In [ ]:
# REPEAT THE PROCESS, BUT THIS TIME SPLIT THE 'df_full_train' INTO 'df_train' AND 'df_val'
#  SINCE WE WANT df_val TO CONTAIN 20% OF THE FULL 'df' DATASET, WE WILL NEED TO RECALCULATE
#  THE 'test_size' PARAMETER - 25% OF THE 'df_full_train' DATASET REPRESENTS 20% OF THE FULL 
#  'df' DATASET
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [ ]:
# CONFIRM DATASET SIZES
print(f"length(df)             {len(df)}")
print(f"\tdf_train df_val\tdf_test")
print(f"\t{len(df_train)} \t {len(df_val)} \t {len(df_test)}")
print(f"Sum of split datasets: {len(df_train) + len(df_val) + len(df_test)}")


In [ ]:
# RESET INDEX VALUES FOR ALL SPLIT DATASETS
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

In [ ]:
# EXTRACT TARGET / LABEL FROM EACH DATASET
y_train  = df_train.churn.values
y_val    = df_val.churn.values
y_test   = df_test.churn.values

In [ ]:
# NOW WE DELETE THE 'churn' COLUMNS FROM FEATURE DATASETS
#  ** DO NOT DELETE BEFORE TARGET / LABEL VECTORS HAVE BEEN CREATED **
del df_train['churn']
del df_val['churn']
del df_test['churn']